[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<USERNAME>/<REPO>/blob/main/UD4_Practico_Optimizadores_JAX.ipynb)
markdown
markdown
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<USERNAME>/<REPO>/blob/main/UD4_Practico_Optimizadores_JAX.ipynb)
\n
Replace `<USERNAME>/<REPO>` with your GitHub user and repository path to make the badge open this notebook directly in Colab.
markdown
markdown
**One-line install (copy for students):**

```bash
mamba create -n ud4-optimizers -y python=3.10 && conda activate ud4-optimizers && mamba install -y -c conda-forge numpy matplotlib scikit-learn && mamba install -n ud4-optimizers -y pytorch torchvision cpuonly -c pytorch

# Alternativa pip:
pip install -r requirements.txt
```
**Instalación local (recomendado con conda/mamba)**:

- Crear entorno con `mamba`/`conda` y activar:
```bash
mamba create -n ud4-optimizers -y python=3.10
conda activate ud4-optimizers
```
- Dependencias principales (CPU):
```bash
mamba install -y -c conda-forge numpy matplotlib scikit-learn
# Optax/JAX (CPU):
pip install optax jax jaxlib
```
- Para ejecutar con GPU instala la rueda `jaxlib` con soporte CUDA y la variante de PyTorch/torchvision para CUDA si las usas.
- Alternativa `pip`: `pip install -r requirements.txt` (ajusta `requirements.txt` según el entorno).

In [ ]:
# Ejemplo numérico: SGD vs Momentum vs Adam (toy) — usando numpy para claridad
import numpy as np

def g(theta):
    return 2*theta

theta0 = 1.0

# SGD
eta_sgd = 0.1
theta_sgd = theta0 - eta_sgd * g(theta0)

# Momentum (dos pasos)
gamma = 0.9
v0 = 0.0
v1 = gamma*v0 + eta_sgd * g(theta0)
theta_mom1 = theta0 - v1
v2 = gamma*v1 + eta_sgd * g(theta_mom1)
theta_mom2 = theta_mom1 - v2

# Adam (primer paso aproximado)
eta_adam = 0.01
beta1 = 0.9
beta2 = 0.999
eps = 1e-8
m1 = (1-beta1)*g(theta0)
v1_adam = (1-beta2)*(g(theta0)**2)
# correcciones de sesgo (primer paso)
mhat = m1/(1-beta1)
vhat = v1_adam/(1-beta2)
delta = eta_adam * mhat / (np.sqrt(vhat) + eps)
theta_adam = theta0 - delta

print('theta0 =', theta0)
print('SGD step ->', theta_sgd)
print('Momentum step1 ->', theta_mom1, ' step2 ->', theta_mom2)
print('Adam approx step ->', theta_adam)


### Ejemplo numérico rápido

Ejemplo numérico corto (idéntico al del documento teórico) que compara la actualización de un parámetro en $L(\theta)=\theta^2$ para SGD, momentum y Adam (aproximación del primer paso). Ejecuta la celda siguiente para ver los valores.

# Práctico: Optimizadores en JAX (Optax)

[Open in Colab](https://colab.research.google.com/) — sube este notebook a Colab o ábrelo desde Drive.

**Nota Colab:** en Colab cambia Runtime → Change runtime type → GPU (opcional).

Este notebook muestra ejemplos prácticos de optimizadores usando `jax` y `optax`.
Objetivo: entrenar un MLP simple sobre Fashion‑MNIST y comparar `SGD` vs `Adam` (optax).

## Instrucciones y Colab
- En Colab: sube el notebook y activa Runtime → Change runtime type → GPU (opcional).
- Si faltan paquetes, ejecuta la celda de instalación que aparece abajo (en Colab suele ser necesaria la instalación de `jax`/`jaxlib`/`optax`).
- Para pruebas rápidas reduce `EPOCHS` o `BATCH_SIZE`.

In [ ]:
# Celda Colab-friendly: monta Drive (opcional) e instala jax/optax si faltan
import importlib, sys, subprocess
IN_COLAB = importlib.util.find_spec('google.colab') is not None
if IN_COLAB:
    print('Entorno: Colab detectado')
    try:
        from google.colab import drive
        print('Montando Google Drive en /content/drive...')
        drive.mount('/content/drive')
    except Exception as e:
        print('No se pudo montar Drive:', e)
else:
    print('No se detectó Colab. Si vas a usar Colab, sube/abre el notebook allí.')

# Instalación recomendada: optax siempre; jax/jaxlib pueden requerir rueda específica en Colab GPU.
def pip_install(packages):
    print('Instalando:', packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

need = []
for p in ('optax',):
    if importlib.util.find_spec(p) is None:
        need.append(p)
# jax/jaxlib: intentar instalar paquete genérico (funciona en Colab CPU); para GPU puede requerir rueda específica
if importlib.util.find_spec('jax') is None:
    need.append('jax')
if importlib.util.find_spec('jaxlib') is None:
    need.append('jaxlib')

if need:
    print('Paquetes faltantes:', need)
    try:
        # Intento genérico; en entornos GPU puede necesitar ajustar jaxlib manualmente
        pip_install(need)
    except Exception as e:
        print('Instalación automática falló:', e)
        print('Si estás en Colab con GPU, instala la rueda jaxlib adecuada. Ver: https://github.com/google/jax#pip-installation')
else:
    print('optax/jax ya instalados')

In [ ]:
import jax
import jax.numpy as jnp
import optax
import numpy as np
import matplotlib.pyplot as plt
print('jax', jax.__version__)
print('optax', optax.__version__)
print('jax devices:', jax.devices())


In [ ]:
# Cargar Fashion-MNIST: preferimos torchvision si está disponible, si no, tensorflow como fallback (Colab)
import importlib
np_backend = None
if importlib.util.find_spec('torchvision') is not None:
    from torchvision.datasets import FashionMNIST
    from torchvision import transforms
    ds = FashionMNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
    x_list = [np.array(img).squeeze() for img, _ in ds]
    y_list = [int(lbl) for _, lbl in ds]
    x_train = np.stack(x_list, axis=0)
    y_train = np.array(y_list, dtype='int32')
    ds_test = FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
    x_list = [np.array(img).squeeze() for img, _ in ds_test]
    y_list = [int(lbl) for _, lbl in ds_test]
    x_test = np.stack(x_list, axis=0)
    y_test = np.array(y_list, dtype='int32')
else:
    try:
        from tensorflow.keras.datasets import fashion_mnist
        (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
    except Exception as e:
        raise RuntimeError('Necesitas tensorflow o torchvision para descargar Fashion-MNIST en este notebook o subir los datos manualmente.') from e

# Normalizar y convertir a floats, flatten (ambos caminos producen arrays HxW)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 28*28)
x_test = x_test.reshape(-1, 28*28)
y_train = y_train.astype('int32')
y_test = y_test.astype('int32')
print('Train', x_train.shape, y_train.shape)


In [ ]:
# Convertir a arrays de JAX (DeviceArray)
X_train = jnp.array(x_train)
y_train_j = jnp.array(y_train)
X_test = jnp.array(x_test)
y_test_j = jnp.array(y_test)


In [ ]:
# Modelo MLP usando Equinox (parecido a PyTorch)
import equinox as eqx
import jax
import jax.numpy as jnp

class MLP(eqx.Module):
    linear1: eqx.nn.Linear
    linear2: eqx.nn.Linear
    def __init__(self, in_dim=28*28, hid=128, out=10, *, key):
        k1, k2 = jax.random.split(key)
        self.linear1 = eqx.nn.Linear(in_dim, hid, key=k1)
        self.linear2 = eqx.nn.Linear(hid, out, key=k2)
    def __call__(self, x):
        x = self.linear1(x)
        x = jax.nn.relu(x)
        x = self.linear2(x)
        return x

# Instanciamos el modelo con Equinox
key = jax.random.PRNGKey(0)
model = MLP(key=key)
print(model)

In [ ]:
# Loss y accuracy (usando el modelo Equinox)
def compute_loss(model, X, y):
    logits = model(X)
    one_hot = jax.nn.one_hot(y, num_classes=10)
    loss = optax.softmax_cross_entropy(logits, one_hot).mean()
    return loss

def accuracy(model, X, y):
    preds = jnp.argmax(model(X), axis=-1)
    return jnp.mean(preds == y)


In [ ]:
# Función de entrenamiento por batches (no compilada) y versión jitted del step
import math
BATCH_SIZE = 256
EPOCHS = 3


In [ ]:
# Data loader simple (numpy) y configuración del optimizador (Optax)
def numpy_data_loader(X, y, batch_size, shuffle=True):
    n = X.shape[0]
    idx = np.arange(n)
    if shuffle:
        np.random.shuffle(idx)
    for i in range(0, n, batch_size):
        batch_idx = idx[i:i+batch_size]
        yield jnp.array(X[batch_idx]), jnp.array(y[batch_idx])

# Elige optimizador: 'adam' o 'sgd'
optim_name = 'adam'
if optim_name == 'adam':
    optimizer = optax.adam(1e-3)
else:
    optimizer = optax.sgd(1e-2)

# Inicializamos estado del optimizador sobre los parámetros filtrados
params = eqx.filter(model, eqx.is_array)
opt_state = optimizer.init(params)

# Paso jitted: calcula gradiente sobre los parámetros y aplica actualización
def loss_params(params, model, X, y):
    # reensamblar modelo con los parámetros (arrays) nuevos para evaluar
    model = eqx.tree_at(lambda m: eqx.filter(m, eqx.is_array), model, params)
    logits = model(X)
    one_hot = jax.nn.one_hot(y, num_classes=10)
    return optax.softmax_cross_entropy(logits, one_hot).mean()

@jax.jit
def update(model, opt_state, Xb, yb):
    params = eqx.filter(model, eqx.is_array)
    loss, grads = jax.value_and_grad(lambda p: loss_params(p, model, Xb, yb))(params)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    # reconstruir modelo con nuevos parámetros
    new_model = eqx.tree_at(lambda m: eqx.filter(m, eqx.is_array), model, new_params)
    return new_model, opt_state, loss

In [ ]:
# Bucle de entrenamiento (rápido)
train_losses = []
train_accs = []
for epoch in range(EPOCHS):
    epoch_losses = []
    for Xb, yb in numpy_data_loader(x_train, y_train, BATCH_SIZE, shuffle=True):
        params, opt_state, loss = update(params, opt_state, Xb, yb)
        epoch_losses.append(float(loss))
    tr_loss = float(np.mean(epoch_losses))
    # evaluamos aproximado en una submuestra para rapidez
    sub = min(2000, x_train.shape[0])
    tr_acc = float(accuracy(params, jnp.array(x_train[:sub]), jnp.array(y_train[:sub])))
    train_losses.append(tr_loss)
    train_accs.append(tr_acc)
    print(f'Epoch {epoch+1}/{EPOCHS} loss={tr_loss:.4f} acc~{tr_acc:.4f}')

In [ ]:
# Evaluación y gráficas
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.plot(train_losses, '-o')
plt.title('Train loss')
plt.subplot(1,2,2)
plt.plot(train_accs, '-o')
plt.title('Train acc (subset)')
plt.tight_layout()

# Evaluación final en test (submuestra rápida)
sub_test = min(2000, x_test.shape[0])
test_acc = float(accuracy(params, jnp.array(x_test[:sub_test]), jnp.array(y_test[:sub_test])))
print(f'Test acc (approx, {sub_test} samples) = {test_acc:.4f}')